## 1.Import libraries

In [1]:
import pandas as pd
import numpy as np
import ast

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

##  2 - Load datasets


In [2]:
raw_customer_df = pd.read_csv("../dataset/customer_dataset.csv")
mobile_df = pd.read_csv("../dataset/GSMArena_Cleaned_Dataset.csv")
segmented_df = pd.read_csv("segmentation_outputs/customer_segments.csv")

In [3]:
raw_customer_df.head()

,customer_id,customer_name,age,gender,city,country,mobile_brand_purchased,mobile_model_purchased,purchase_date,purchase_amount_npr,...,wishlist,rating,review,preferred_brand,preferred_category,accessories_purchased,warranty_opted,exchange_history,interaction_channel,last_active_at
0,CUST-000EFD69,Ankit Hamal,75,Male,Hetauda,Nepal,Apple,Apple iPhone XS,2021-07-14,96700.0,...,"[{""item"": ""Apple Apple iPhone XS"", ""added_at"":...",5,Would buy again from this brand,Apple,Battery-focused,"[{""item"": ""Earphones"", ""price_npr"": 642, ""purc...",Yes,[],Mobile App,2024-04-26 12:00:00
1,CUST-000EFD69,Ankit Hamal,75,Male,Hetauda,Nepal,Apple,Apple iPhone 17 Pro Max,2022-10-21,81300.0,...,"[{""item"": ""Apple Apple iPhone 17 Pro Max"", ""ad...",5,"Average phone, nothing special",Apple,Battery-focused,"[{""item"": ""Stylus Pen"", ""price_npr"": 1574, ""pu...",No,[],Daraz,2024-04-26 12:00:00
2,CUST-000EFD69,Ankit Hamal,75,Male,Hetauda,Nepal,Apple,Apple iPhone XS Max,2023-09-05,80900.0,...,"[{""item"": ""Apple Apple iPhone XS Max"", ""added_...",3,Could be better at this price,Apple,Battery-focused,"[{""item"": ""Camera Lens Kit"", ""price_npr"": 4217...",No,[],In-store,2024-04-26 12:00:00
3,CUST-000EFD69,Ankit Hamal,75,Male,Hetauda,Nepal,Apple,Apple iPhone 16,2024-04-26,94000.0,...,"[{""item"": ""Apple Apple iPhone 16"", ""added_at"":...",4,Heating issue during heavy use,Apple,Battery-focused,"[{""item"": ""Earphones"", ""price_npr"": 2369, ""pur...",No,[],WhatsApp,2024-04-26 12:00:00
4,CUST-00415B2B,Sandesh Limbu,58,Male,Gorkha,Nepal,Vivo,vivo iQOO Neo9S Pro+,2021-06-04,46000.0,...,"[{""item"": ""Xiaomi Xiaomi Redmi 12C"", ""added_at...",4,"Average phone, nothing special",Samsung,Gaming,"[{""item"": ""Selfie Stick"", ""price_npr"": 440, ""p...",Yes,[],Daraz,2024-03-11 12:00:00


In [4]:
raw_customer_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16608 entries, 0 to 16607
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   customer_id                  16608 non-null  str    
 1   customer_name                16608 non-null  str    
 2   age                          16608 non-null  int64  
 3   gender                       16608 non-null  str    
 4   city                         16608 non-null  str    
 5   country                      16608 non-null  str    
 6   mobile_brand_purchased       16608 non-null  str    
 7   mobile_model_purchased       16608 non-null  str    
 8   purchase_date                16608 non-null  str    
 9   purchase_amount_npr          16608 non-null  float64
 10  payment_method               16608 non-null  str    
 11  purchase_frequency_per_year  16608 non-null  float64
 12  average_spend_npr            16608 non-null  int64  
 13  browsing_history           

In [5]:
mobile_df.head()

,Brand,Model_Name,Model_URL,Model_Image,Network_Technology,5G_Support,Announced,Status,Dimensions,Weight_g,...,Has_Elite,Has_HDR,Has_Panorama,Has_Color_Spectrum,Flash_Type,GPU_Brand,GPU_Family,GPU_Is_Flagship,Wired_Charging_W_is_imputed,CPU_architecture_is_imputed
0,Acer,Acer Liquid,https://www.gsmarena.com/acer_liquid-2968.php,https://fdn2.gsmarena.com/vv/bigpic/acer-liqui...,GSM / HSPA,No,"2009, October. Released 2009, December",Discontinued,115 x 62.5 x 12.5 mm (4.53 x 2.46 x 0.49 in),135.0,...,0,0,0,0,NaN,Adreno,Adreno,0,True,0
1,Acer,Acer neoTouch,https://www.gsmarena.com/acer_neotouch-2958.php,https://fdn2.gsmarena.com/vv/bigpic/acer-neo-t...,GSM / HSPA,No,"2009, October. Released 2009, October",Discontinued,118.6 x 63 x 12 mm (4.67 x 2.48 x 0.47 in),130.0,...,0,0,0,0,LED,Adreno,Adreno,0,True,0
2,Acer,Acer beTouch E200,https://www.gsmarena.com/acer_betouch_e200-296...,https://fdn2.gsmarena.com/vv/bigpic/acer-be-to...,GSM / HSPA,No,"2009, October. Released 2009, October",Discontinued,110 x 53.5 x 15.4 mm (4.33 x 2.11 x 0.61 in),146.0,...,0,0,0,0,NaN,Other,Other,0,True,0
3,Acer,Acer beTouch E100,https://www.gsmarena.com/acer_betouch_e100-295...,https://fdn2.gsmarena.com/vv/bigpic/acer-be-to...,GSM / HSPA,No,"2009, October. Released 2009, October",Discontinued,113 x 56 x 12.8 mm (4.45 x 2.20 x 0.50 in),118.0,...,0,0,0,0,NaN,Other,Other,0,True,0
4,Acer,Acer beTouch E101,https://www.gsmarena.com/acer_betouch_e101-296...,https://fdn2.gsmarena.com/vv/bigpic/acer-be-to...,GSM,No,"2009, October. Released 2009, October",Discontinued,113 x 56 x 12.8 mm (4.45 x 2.20 x 0.50 in),118.0,...,0,0,0,0,NaN,Other,Other,0,True,0


In [6]:
mobile_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8500 entries, 0 to 8499
Data columns (total 100 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Brand                        8500 non-null   str    
 1   Model_Name                   8500 non-null   str    
 2   Model_URL                    8500 non-null   str    
 3   Model_Image                  8500 non-null   str    
 4   Network_Technology           8500 non-null   str    
 5   5G_Support                   8500 non-null   str    
 6   Announced                    8500 non-null   str    
 7   Status                       8500 non-null   str    
 8   Dimensions                   8500 non-null   str    
 9   Weight_g                     8500 non-null   float64
 10  SIM_Type                     8500 non-null   str    
 11  Display_Type                 8500 non-null   str    
 12  Refresh_Rate_Hz              8500 non-null   float64
 13  Display_Size_inch           

In [7]:
segmented_df.head()

,customer_id,customer_name,age,gender,city,country,preferred_brand,preferred_category,payment_method,interaction_channel,...,tenure_days,wishlist_count,wishlist_conversion_rate,accessory_count,accessory_spend_npr,browsing_count,brand_loyal,cluster_id,segment_name,recommendation_persona
0,CUST-000EFD69,Ankit Hamal,75,Male,Hetauda,Nepal,Apple,Battery-focused,Bank Transfer,Daraz,...,1017,0,0.0,0,0.0,30,1,2,Luxury Xiaomi Battery-focused - Frequent but...,Luxury Xiaomi Battery-focused - Frequent but...
1,CUST-00415B2B,Sandesh Limbu,58,Male,Gorkha,Nepal,Samsung,Gaming,Bank Transfer,Daraz,...,1011,0,0.0,0,0.0,36,0,2,Luxury Xiaomi Battery-focused - Frequent but...,Luxury Xiaomi Battery-focused - Frequent but...
2,CUST-004C2405,Ambika Simkhada,51,Female,Myagdi,Nepal,Apple,Mid-range,Cash,Daraz,...,1413,0,0.0,0,0.0,47,0,2,Luxury Xiaomi Battery-focused - Frequent but...,Luxury Xiaomi Battery-focused - Frequent but...
3,CUST-0061E821,Shankar Gurung,56,Male,Rasuwa,Nepal,Xiaomi,Battery-focused,Bank Transfer,Instagram DM,...,679,0,0.0,0,0.0,23,0,0,Premium Xiaomi Budget - Frequent but Lapsing,Premium Xiaomi Budget - Frequent but Lapsing
4,CUST-00709A32,Rojina Sapkota,56,Female,Darchula,Nepal,Xiaomi,Foldable,Debit Card,Website,...,1370,0,0.0,0,0.0,37,0,2,Luxury Xiaomi Battery-focused - Frequent but...,Luxury Xiaomi Battery-focused - Frequent but...


In [8]:
# keep the most recent purchase per customer
raw_customer_df["purchase_date"] = pd.to_datetime(raw_customer_df["purchase_date"], errors="coerce")
raw_customer_df = raw_customer_df.sort_values("purchase_date")
customer_df = raw_customer_df.groupby("customer_id").tail(1).reset_index(drop=True)

# add the Cluster column from the segmentation output
customer_df = customer_df.merge(
    segmented_df[["customer_id", "cluster_id"]],
    on="customer_id",
    how="left"
)
customer_df = customer_df.rename(columns={"cluster_id": "Cluster"})

print(customer_df.shape)
customer_df[["customer_id", "preferred_brand", "mobile_brand_purchased", "Cluster"]].head()

(4557, 25)


,customer_id,preferred_brand,mobile_brand_purchased,Cluster
0,CUST-B3436D3E,Vivo,Vivo,1
1,CUST-BDB114D7,Samsung,Samsung,0
2,CUST-EE592206,Xiaomi,Huawei,1
3,CUST-4397AA7F,Samsung,Samsung,1
4,CUST-DCA5DAE2,Vivo,Motorola,0


## 3 - Keep only useful phone columns


In [9]:
phone_features = [
    "Brand",
    "Model_Name",
    "RAM_GB",
    "Storage_GB",
    "Battery_mAh",
    "Main_Camera_MP",
    "Price_EUR",
    "Display_Size_inch",
    "5G_Support",
    "Chipset_Brand"
]

phones = mobile_df[phone_features].copy()
phones = phones.dropna().reset_index(drop=True)
phones.head()

,Brand,Model_Name,RAM_GB,Storage_GB,Battery_mAh,Main_Camera_MP,Price_EUR,Display_Size_inch,5G_Support,Chipset_Brand
0,Acer,Acer Liquid,1.0,8.0,1350.0,5.00,90.0,3.5,No,Qualcomm
1,Acer,Acer neoTouch,1.0,8.0,1350.0,5.00,210.0,3.8,No,Qualcomm
2,Acer,Acer beTouch E200,1.0,8.0,1140.0,3.15,190.0,3.0,No,Qualcomm
3,Acer,Acer beTouch E100,1.0,8.0,1140.0,2.00,180.0,3.2,No,Qualcomm
4,Acer,Acer beTouch E101,1.0,8.0,1140.0,2.00,170.0,3.2,No,Qualcomm


## 4 - Preprocess phone data



In [10]:
cat_cols = ["Brand", "Chipset_Brand", "5G_Support"]
num_cols = ["RAM_GB", "Storage_GB", "Battery_mAh", "Main_Camera_MP", "Price_EUR", "Display_Size_inch"]

phones_encoded = pd.get_dummies(phones[cat_cols], drop_first=False)

scaler = StandardScaler()
phones_scaled_num = pd.DataFrame(
    scaler.fit_transform(phones[num_cols]),
    columns=num_cols,
    index=phones.index
)

phone_matrix = pd.concat([phones_scaled_num, phones_encoded], axis=1)
phone_matrix.head()

,RAM_GB,Storage_GB,Battery_mAh,Main_Camera_MP,Price_EUR,Display_Size_inch,Brand_Acer,Brand_Alcatel,Brand_Allview,Brand_Apple,...,Chipset_Brand_Qualcomm,Chipset_Brand_Rockchip,Chipset_Brand_ST-Ericsson,Chipset_Brand_Samsung,Chipset_Brand_Spreadtrum,Chipset_Brand_Texas_Instruments,Chipset_Brand_Unisoc,Chipset_Brand_Unknown,5G_Support_No,5G_Support_Yes
0,-0.979031,-0.806579,-1.154187,-0.737750,-0.730630,-1.534729,True,False,False,False,...,True,False,False,False,False,False,False,False,True,False
1,-0.979031,-0.806579,-1.154187,-0.737750,-0.164903,-1.360629,True,False,False,False,...,True,False,False,False,False,False,False,False,True,False
2,-0.979031,-0.806579,-1.237605,-0.797319,-0.259190,-1.824896,True,False,False,False,...,True,False,False,False,False,False,False,False,True,False
3,-0.979031,-0.806579,-1.237605,-0.834349,-0.306334,-1.708829,True,False,False,False,...,True,False,False,False,False,False,False,False,True,False
4,-0.979031,-0.806579,-1.237605,-0.834349,-0.353478,-1.708829,True,False,False,False,...,True,False,False,False,False,False,False,False,True,False


## 5 - Parse browsing history and wishlist



In [11]:
def parse_list(value):
    try:
        return ast.literal_eval(value)
    except Exception:
        return []

def get_browsed_items(value):
    items = parse_list(value)
    return [d.get("item", "") for d in items if isinstance(d, dict)]

def get_wishlist_phones(value):
    items = parse_list(value)
    return [d.get("item", "") for d in items if isinstance(d, dict)]

customer_df["browsed_items"] = customer_df["browsing_history"].apply(get_browsed_items)
customer_df["wishlist_phones"] = customer_df["wishlist"].apply(get_wishlist_phones)

customer_df[["customer_id", "browsed_items", "wishlist_phones"]].head()

,customer_id,browsed_items,wishlist_phones
0,CUST-B3436D3E,"[Battery Phones, Samsung, Vivo, Huawei, Laptop...",[]
1,CUST-BDB114D7,"[5G Phones, Huawei, Vivo, New Arrivals, Honor,...",[]
2,CUST-EE592206,"[Comparison, Xiaomi, Tecno, Oppo, Tablets, Sam...",[]
3,CUST-4397AA7F,"[Apple, Oppo, Budget Phones, Xiaomi, Vivo, Inf...",[]
4,CUST-DCA5DAE2,"[Samsung, Apple, Mid-range Phones, Sale Items,...",[]


##  6 - Build a simple customer preference vector

In [12]:
all_brands = set(phones["Brand"].unique())

def brand_from_browsing(items):
    for x in items:
        if x in all_brands:
            return x
    return None

customer_df["brand_from_browsing"] = customer_df["browsed_items"].apply(brand_from_browsing)

npr_to_eur = 1 / 150
customer_df["budget_eur"] = customer_df["average_spend_npr"] * npr_to_eur

customer_df[["customer_id", "preferred_brand", "mobile_brand_purchased",
            "brand_from_browsing", "budget_eur"]].head()

,customer_id,preferred_brand,mobile_brand_purchased,brand_from_browsing,budget_eur
0,CUST-B3436D3E,Vivo,Vivo,Samsung,1297.773333
1,CUST-BDB114D7,Samsung,Samsung,Huawei,543.773333
2,CUST-EE592206,Xiaomi,Huawei,Xiaomi,1583.106667
3,CUST-4397AA7F,Samsung,Samsung,Apple,738.440000
4,CUST-DCA5DAE2,Vivo,Motorola,Samsung,596.886667


In [13]:
def build_customer_vector(row):
    vec = pd.Series(0.0, index=phone_matrix.columns)

    # brand preferences (one hot like)
    for col in ["Brand_" + str(row["preferred_brand"]),
                "Brand_" + str(row["mobile_brand_purchased"]),
                "Brand_" + str(row["brand_from_browsing"])]:
        if col in vec.index:
            vec[col] += 1.0

    if any("5G" in str(x).upper() for x in row["browsed_items"]):
        if "5G_Support_Yes" in vec.index:
            vec["5G_Support_Yes"] += 1.0

    budget_scaled = scaler.transform(
        pd.DataFrame([[0, 0, 0, 0, row["budget_eur"], 0]], columns=num_cols)
    )[0][num_cols.index("Price_EUR")]
    vec["Price_EUR"] = budget_scaled

    return vec

customer_vectors = customer_df.apply(build_customer_vector, axis=1)
customer_vectors.head()

,RAM_GB,Storage_GB,Battery_mAh,Main_Camera_MP,Price_EUR,Display_Size_inch,Brand_Acer,Brand_Alcatel,Brand_Allview,Brand_Apple,...,Chipset_Brand_Qualcomm,Chipset_Brand_Rockchip,Chipset_Brand_ST-Ericsson,Chipset_Brand_Samsung,Chipset_Brand_Spreadtrum,Chipset_Brand_Texas_Instruments,Chipset_Brand_Unisoc,Chipset_Brand_Unknown,5G_Support_No,5G_Support_Yes
0,0.0,0.0,0.0,0.0,4.963294,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.408638,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.0,0.0,0.0,0.0,6.308468,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,2.326374,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.659035,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 7 - Cluster based personalization

In [14]:
cluster_brand = (
    customer_df.groupby("Cluster")["preferred_brand"]
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else None)
    .to_dict()
)
cluster_brand

{0: 'Samsung', 1: 'Apple', 2: 'Samsung'}

## 8 - Content based recommendation using cosine similarity



In [15]:
def recommend_for_row(row, top_n=5, cluster_boost=0.05):
    cust_vec = customer_vectors.loc[row.name].values.reshape(1, -1)
    sims = cosine_similarity(cust_vec, phone_matrix.values)[0]

    cluster = row["Cluster"]
    boost_brand = cluster_brand.get(cluster, None)
    if boost_brand is not None:
        boost = np.where(phones["Brand"].values == boost_brand, cluster_boost, 0.0)
    else:
        boost = np.zeros(len(phones))

    final_score = sims + boost
    top_idx = np.argsort(final_score)[::-1][:top_n]

    return phones.iloc[top_idx][
        ["Brand", "Model_Name", "Price_EUR", "RAM_GB", "Battery_mAh", "Main_Camera_MP"]
    ].assign(similarity=final_score[top_idx])

## 9 - Reusable function

In [16]:
def recommend_mobile(customer_id, top_n=5):
    matches = customer_df[customer_df["customer_id"] == customer_id]
    if matches.empty:
        return f"Customer {customer_id} not found"
    row = matches.iloc[0]
    print(f"Customer: {row['customer_id']}")
    print(f"Preferred brand: {row['preferred_brand']}, Purchased: {row['mobile_brand_purchased']}")
    print(f"Cluster: {row['Cluster']}, Category: {row['preferred_category']}")
    print(f"Average spend (NPR): {row['average_spend_npr']}")
    print()
    return recommend_for_row(row, top_n=top_n)

## 10 - Test the system

In [17]:
recommend_mobile(customer_df.iloc[0]["customer_id"])

Customer: CUST-B3436D3E
Preferred brand: Vivo, Purchased: Vivo
Cluster: 1, Category: Mid-range
Average spend (NPR): 194666



,Brand,Model_Name,Price_EUR,RAM_GB,Battery_mAh,Main_Camera_MP,similarity
1865,Honor,Honor Magic Vs,2940.0,8.0,5000.0,54.0,0.887404
687,Asus,Asus Zenfone 10,2140.0,8.0,4300.0,50.0,0.885469
7105,Vivo,vivo X100,2000.0,12.0,5000.0,50.0,0.883893
4680,Oppo,Oppo Find X,1950.0,8.0,3730.0,16.0,0.881561
7154,Vivo,vivo X Fold3 Pro,2000.0,12.0,5700.0,50.0,0.876791


In [18]:
recommend_mobile(customer_df.iloc[1]["customer_id"])

Customer: CUST-BDB114D7
Preferred brand: Samsung, Purchased: Samsung
Cluster: 0, Category: Budget
Average spend (NPR): 81566



,Brand,Model_Name,Price_EUR,RAM_GB,Battery_mAh,Main_Camera_MP,similarity
5821,Samsung,Samsung Galaxy Z Flip7 FE,734.500000,8.0,4000.0,50.0,0.741652
5788,Samsung,Samsung Galaxy XCover7 Pro,423.990000,6.0,4350.0,50.0,0.734291
5838,Samsung,Samsung Galaxy S25 FE,459.990000,8.0,4900.0,50.0,0.690453
5765,Samsung,Samsung Galaxy S24,429.680000,8.0,4000.0,50.0,0.684401
5873,Samsung,Samsung Galaxy S6 Plus,779.807339,4.0,3000.0,16.0,0.664635


In [19]:
recommend_mobile(customer_df.iloc[2]["customer_id"])

Customer: CUST-EE592206
Preferred brand: Xiaomi, Purchased: Huawei
Cluster: 1, Category: Rugged
Average spend (NPR): 237466



,Brand,Model_Name,Price_EUR,RAM_GB,Battery_mAh,Main_Camera_MP,similarity
1865,Honor,Honor Magic Vs,2940.0000,8.0,5000.0,54.0,0.917380
687,Asus,Asus Zenfone 10,2140.0000,8.0,4300.0,50.0,0.915379
4680,Oppo,Oppo Find X,1950.0000,8.0,3730.0,16.0,0.911339
2660,Huawei,Huawei Mate XT Ultimate,3370.3713,16.0,5600.0,50.0,0.904564
7637,Xiaomi,Xiaomi Mi 11 Ultra,1500.0000,8.0,5000.0,50.0,0.897106


In [20]:
customer_df.iloc[2]["customer_id"]

'CUST-EE592206'